In [2]:
print("test")

test


In [15]:
%pip install pyarrow

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/layers/paketo-buildpacks_poetry-install/poetry-venv/datascience-python-renku-dependencies-xS3fZVNL-py3.13/lib/python3.13/site-packages/pip/__main__.py", line 8, in <module>
    if sys.path[0] in ("", os.getcwd()):
                           ~~~~~~~~~^^
FileNotFoundError: [Errno 2] No such file or directory
Note: you may need to restart the kernel to use updated packages.


In [2]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 57 bits virtual
  Byte Order:                Little Endian
CPU(s):                      16
  On-line CPU(s) list:       0-15
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) Platinum 8370C CPU @ 2.80GHz
    CPU family:              6
    Model:                   106
    Thread(s) per core:      2
    Core(s) per socket:      8
    Socket(s):               1
    Stepping:                6
    CPU(s) scaling MHz:      99%
    CPU max MHz:             2800.0000
    CPU min MHz:             800.0000
    BogoMIPS:                5586.87
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology tsc_rel

In [3]:
%pip install pandas



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd

df = pd.read_csv("../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv", sep=";")

df.head()



,GP-Nr,PLZ,Ort,Kanton,WärmePumpe,PV,PV-Leistung in kWp,Batterie/Speicher,Ladestation für Elektrofahrzeuge,Wärmepumpenboiler,Datum Unterschrift,geplanter Baustart,Übergabe,InBetrieb-Datum
0,698970,5452.0,Oberrohrdorf,AG,-,x,NaN,-,-,-,18.04.2008,NaN,NaN,NaN
1,NaN,5024.0,Küttigen,AG,-,-,NaN,-,-,-,01.01.2017,NaN,NaN,NaN
2,196344,5732.0,Zetzwil,AG,-,-,NaN,-,-,-,01.01.2017,NaN,NaN,21.06.2017
3,570021,5023.0,Biberstein,AG,x,-,NaN,-,-,-,01.07.2017,NaN,NaN,24.11.2017
4,716671,8962.0,Bergdietikon,AG,x,x,NaN,x,-,-,24.08.2017,NaN,23.11.2017,04.12.2017


/home/renku/work/OSNOVA/dataanalysis


In [ ]:
from pathlib import Path
import pandas as pd

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)
    parquet_file = target / relative.with_suffix(".parquet")

    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        # Your GIGI file uses ;
        df = pd.read_csv(csv_file, sep=";")

        # Clean column names
        df.columns = df.columns.str.strip()

        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

Converted: ../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv -> ../../store/parquet_data/HackDays2026 - GIGI.parquet
Converted: ../../aew-data/test-blob/input_data/Zähler-GP.csv -> ../../store/parquet_data/Zähler-GP.parquet
Converted: ../../aew-data/test-blob/input_data/mpid_zähler_mapping.csv -> ../../store/parquet_data/mpid_zähler_mapping.parquet
Converted: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv -> ../../store/parquet_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.parquet
Converted: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv -> ../../store/parquet_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.parquet
Converted: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv -> ../../store/parquet_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.parquet
Converted: ../../aew-data/test-blob/input_data/2

In [3]:
from pathlib import Path
import pandas as pd

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

skip_years = {"2025", "2026"}

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)

    # Skip anything inside 2025 or 2026 folders
    if any(part in skip_years for part in relative.parts):
        print(f"SKIPPED YEAR: {csv_file}")
        continue

    parquet_file = target / relative.with_suffix(".parquet")
    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    # Skip files that were already converted
    if parquet_file.exists():
        print(f"ALREADY EXISTS: {parquet_file}")
        continue

    try:
        df = pd.read_csv(csv_file, sep=";")
        df.columns = df.columns.str.strip()

        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

print("done")

ALREADY EXISTS: ../../store/parquet_data/HackDays2026 - GIGI.parquet
ALREADY EXISTS: ../../store/parquet_data/Zähler-GP.parquet
ALREADY EXISTS: ../../store/parquet_data/mpid_zähler_mapping.parquet
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Juli 2026/LG_AIM2Hackerdays_kWh_20260824_102023.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Juni 2026/LG_AIM2Hackerdays_kWh_20260729_062053.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/Mai 2026/LG_AIM2Hackerdays_kWh_20260728_195653.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data/2026/März 2026/LG_AIM2Hackerdays_kWh_20260728_063802.csv
SKIPPED YEAR: ../../aew-data/test-blob/input_data

In [4]:
from pathlib import Path
import pandas as pd
import csv

source = Path("../../aew-data/test-blob/input_data")
target = Path("../../store/parquet_data")

wanted_years = {"2023", "2024"}

for csv_file in source.rglob("*.csv"):
    relative = csv_file.relative_to(source)

    # Only process 2023 and 2024
    year = next(
        (part for part in relative.parts if part in wanted_years),
        None
    )

    if year is None:
        print(f"SKIPPED: {csv_file}")
        continue

    parquet_file = target / relative.with_suffix(".parquet")
    parquet_file.parent.mkdir(parents=True, exist_ok=True)

    # 2024 files can be skipped if already converted.
    # 2023 files are intentionally overwritten because we need to fix them.
    if year == "2024" and parquet_file.exists():
        print(f"ALREADY EXISTS: {parquet_file}")
        continue

    try:
        if year == "2023":
            # Read header ourselves
            with open(csv_file, "r", encoding="utf-8-sig") as f:
                reader = csv.reader(f, delimiter=";")
                header = next(reader)

            # Remove trailing empty header caused by final ;
            if header and header[-1] == "":
                header = header[:-1]

            # Read data without trusting the malformed header
            df = pd.read_csv(
                csv_file,
                sep=";",
                header=None,
                skiprows=1
            )

            # Remove trailing empty column caused by final ;
            if df.iloc[:, -1].isna().all():
                df = df.iloc[:, :-1]

            # 2023 has an extra second column.
            # Remove it.
            df = df.drop(columns=df.columns[1])

            # Verify that everything now lines up
            if len(df.columns) != len(header):
                raise ValueError(
                    f"Column mismatch after 2023 fix: "
                    f"{len(df.columns)} data columns vs "
                    f"{len(header)} header columns"
                )

            df.columns = header

            print(
                f"2023 FIX: removed extra identifier column "
                f"from {csv_file.name}"
            )

        else:
            # Normal 2024 structure
            df = pd.read_csv(csv_file, sep=";")

            # Remove completely empty trailing column, if present
            unnamed = [
                col for col in df.columns
                if str(col).startswith("Unnamed:")
            ]

            if unnamed:
                df = df.drop(columns=unnamed)

        # Clean whitespace from column names
        df.columns = df.columns.str.strip()

        df.to_parquet(
            parquet_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"Converted: {csv_file} -> {parquet_file}")

    except Exception as e:
        print(f"FAILED: {csv_file}")
        print(e)

SKIPPED: ../../aew-data/test-blob/input_data/HackDays2026 - GIGI.csv
SKIPPED: ../../aew-data/test-blob/input_data/Zähler-GP.csv
SKIPPED: ../../aew-data/test-blob/input_data/mpid_zähler_mapping.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Juli 2026/LG_AIM2Hackerdays_kWh_20260824_102023.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Juni 2026/LG_AIM2Hackerdays_kWh_20260729_062053.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/Mai 2026/LG_AIM2Hackerdays_kWh_20260728_195653.csv
SKIPPED: ../../aew-data/test-blob/input_data/2026/März 2026/LG_AIM2Hackerdays_kWh_20260728_063802.csv
SKIPPED: ../../aew-data/test-blob/input_data/2025/April 2025/LG_AIM2Hackerdays_kWh_2